#  Document Similarity Tool
### Day 13 –  Bag of Words + TF-IDF | Building Document Similarity Tool

---

**What I will build:**
- Understand and implement **Bag of Words (BoW)**
- Implement **TF-IDF** 
- Build a **Document Similarity Tool** using cosine similarity

**Concepts covered:**
| Concept | Description |
|---|---|
| Bag of Words | Represent text as word frequency vectors |
| TF-IDF | Weight words by importance across documents |
| Cosine Similarity | Measure how similar two vectors are |


## 📦 Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import re
from collections import Counter

# sklearn for comparison later
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


##  Step 2 — Define Sample Documents

We will use 5 sample documents covering different topics.
The tool should tell us which documents are most similar to each other.


In [ ]:
documents = [
    "The cat sat on the mat",
    "The dog sat on the floor",
    "Machine learning is part of artificial intelligence",
    "Deep learning uses neural networks",
    "The cat and dog are common pets",
]

# Print them out to see
for i, doc in enumerate(documents):
    print(f"Doc {i+1}: {doc}")


## Step 4 — Bag of Words (BoW)

**Bag of Words** represents each document as a vector of word counts.

- Build a **vocabulary** (all unique words across all documents)
- For each document, count how many times each vocabulary word appears
- The result is a **document × word matrix**

>  BoW ignores word order — it only cares about **frequency**


In [ ]:
# CountVectorizer converts text into word count vectors
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(documents)

# See all the unique words found
print("Vocabulary (unique words):")
print(vectorizer.get_feature_names_out())


###  See the BoW Table

In [ ]:
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("Bag of Words Matrix (word counts):")
print(bow_df)


##  Step 5 — TF-IDF from Scratch

**BoW problem:** Common words like "sat" appear in many documents, making them seem similar even if the content is different.

**TF-IDF solution:** Weight each word by how *unique* it is to a document.

$$TF(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

$$IDF(t) = \log\left(\frac{N + 1}{df(t) + 1}\right) + 1 \quad \text{(smoothed)}$$

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

Where:
- **N** = total number of documents
- **df(t)** = number of documents containing term t
- Smoothing prevents division by zero


In [ ]:
# TF-IDF gives less importance to common words like "the", "is"
# and more importance to unique words

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("TF-IDF Matrix:")
print(tfidf_df)


##  Step 6 — Cosine Similarity

We compare documents by measuring the **angle** between their TF-IDF vectors.

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

- Result is between **0** (completely different) and **1** (identical)
- It is **length-independent** — a short and long document about the same topic still score high


In [ ]:
# Cosine similarity: how similar are two documents?
# 1.0 = identical, 0.0 = completely different

sim_matrix = cosine_similarity(tfidf_matrix)

sim_df = pd.DataFrame(
    sim_matrix.round(3),
    index=[f"Doc {i+1}" for i in range(len(documents))],
    columns=[f"Doc {i+1}" for i in range(len(documents))]
)

print("Similarity Matrix:")
print(sim_df)


### Visualise the Similarity Matrix

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(sim_matrix, cmap="Blues")
plt.colorbar(label="Similarity Score")
plt.xticks(range(5), [f"Doc {i+1}" for i in range(5)])
plt.yticks(range(5), [f"Doc {i+1}" for i in range(5)])
plt.title("Document Similarity Heatmap")

# Add scores inside the boxes
for i in range(5):
    for j in range(5):
        plt.text(j, i, round(sim_matrix[i][j], 2), ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.show()


##  Step 7 — Finding Most Similar Document to a Query



In [ ]:
query = ["cat and dog sitting at home"]

# Convert query using the same TF-IDF vocabulary
query_vec = tfidf_vectorizer.transform(query)

# Compare query with all documents
scores = cosine_similarity(query_vec, tfidf_matrix)[0]

print(f"Query: '{query[0]}'\n")
for i, score in enumerate(scores):
    print(f"  Doc {i+1}: {round(score, 3)}  →  {documents[i]}")

print(f"\nMost similar: Doc {scores.argmax() + 1}")

# Trying My own Query 

In [ ]:
my_query = ["neural networks and machine learning"]

my_vec = tfidf_vectorizer.transform(my_query)
my_scores = cosine_similarity(my_vec, tfidf_matrix)[0]

print(f"Query: '{my_query[0]}'\n")
for i, score in enumerate(my_scores):
    print(f"  Doc {i+1} (score: {round(score, 3)}) → {documents[i]}")

best = my_scores.argmax()
print(f"\n Best match: Doc {best+1} — '{documents[best]}'")

##  Summary — What I Built

| Step | What we did |
|---|---|
| Bag of Words | Count matrix — each word's frequency per document |
| TF-IDF | Weight words by rarity across documents |
| Cosine Similarity | Measure angle between two document vectors |
| Similarity Tool |  fit on docs, query to find most similar |

###  Key Takeaways
- **BoW** is simple but treats all words equally — common words dominate
- **TF-IDF** fixes this by down-weighting frequent words across documents
- **Cosine similarity** is robust to document length differences
- **Real NLP pipelines** (search engines, recommendation systems) are built on exactly these ideas


